# 🚀 Mizan-Rerank-v3-Turbo: 10,000+ Multi-Scholar Dataset Re-Scoring & Side-by-Side Benchmark
### Evaluating the Next-Generation ModernBERT Arabic Reranker against Mizan V2 & Qwen 0.6B Embedder

---

### 📌 Overview
This notebook evaluates the newly released **`ALJIACHI/Mizan-Rerank-v3-Turbo`** on our comprehensive **10,044 multi-scholar candidate pool** (spanning 942 queries and 7 scholarly sources: Ibn Baz, Al-Lajnah Ad-Daimah, Ibn Uthaymeen, Saleh Al-Fawzan, Abdullah al-Manee, Saleh bin Humaid, Abdulaziz Aal ash-Sheikh).

#### Key Upgrades in Mizan-Rerank-v3-Turbo:
* **Architecture:** Built on **ModernBERT** with native Rotary Positional Embeddings (RoPE), Unpadding, and FlashAttention-2 compatibility.
* **Context Length:** Supports up to **8,192 tokens** (vs 1,024/2,048 in V2), allowing full fatwa text without truncation.
* **Efficiency:** Compact ~150M parameters (~half the size of V2), yielding ~2× to 3× higher throughput (>2,000 pairs/sec on GPU).
* **Target:** Higher discriminative separation on complex Arabic jurisprudential distinctions and hard negatives.

---

### 📋 Notebook Pipeline
1. **Environment Setup & GPU Check:** PyTorch, CUDA, and library dependencies.
2. **Ingest 10k+ Multi-Scholar Dataset:** Automatically locates or unifies the 10,044 candidate pool.
3. **Load `ALJIACHI/Mizan-Rerank-v3-Turbo`:** Initialize ModernBERT Cross-Encoder on GPU.
4. **Mass Inference & Throughput Profiling:** Score all 10,044 pairs with batching and measure throughput.
5. **Comprehensive 3-Way Benchmark:** Direct head-to-head comparison:
   * **Embedder:** `Qwen/Qwen3-Embedding-0.6B`
   * **Reranker V2:** `ALJIACHI/Mizan-Rerank-V2`
   * **Reranker V3 Turbo:** `ALJIACHI/Mizan-Rerank-v3-Turbo`
   * *Metrics:* Hit@1, Hit@3, Hit@5, Hit@10, P@k, R@k, NDCG@k, MRR, MAP, ROC-AUC, PR-AUC.
6. **End-to-End Live Retrieval + V3 Reranking:** Run semantic retrieval on the multi-scholar corpus and rerank top fatwas.
7. **Packaging & Export:** Save the rescored 10k dataset and evaluation metrics for immediate download.


In [1]:
# 1. Environment & GPU Verification
!nvidia-smi

import os, sys, time, json, gc
from pathlib import Path
import torch

print(f"\nPyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Primary GPU:     {device_name} ({vram_gb:.2f} GB VRAM)")
    print(f"Device Count:    {torch.cuda.device_count()}")
    device = "cuda"
else:
    print("WARNING: CUDA GPU not detected. Running on CPU (slower).")
    device = "cpu"


Tue Oct 27 20:24:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Install Required Dependencies
!pip install -q "sentence-transformers>=3.1.0" "transformers>=4.48.0" huggingface_hub scikit-learn pandas numpy tabulate tqdm
print("✓ Dependencies installed successfully.")


✓ Dependencies installed successfully.


In [3]:
# 3. Load Multi-Scholar 10k Dataset & Corpus Embeddings
import pandas as pd
import numpy as np
import zipfile
from huggingface_hub import hf_hub_download

# Search directories for the 10k dataset or bundles
SEARCH_DIRS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("."),
    Path("../data"),
    Path("data"),
]

def find_file(filename: str):
    for root in SEARCH_DIRS:
        if root.exists():
            for p in root.rglob(filename):
                return p
    return None

# Attempt 1: Load pre-unified 10k dataset if present
dataset_csv = find_file("dataset_10k_multi_scholar.csv")
dataset_json = find_file("dataset_10k_multi_scholar.json")

df_10k = None

if dataset_csv and dataset_csv.exists():
    print(f"✓ Found unified 10k dataset at: {dataset_csv}")
    df_10k = pd.read_csv(dataset_csv)
elif dataset_json and dataset_json.exists():
    print(f"✓ Found unified 10k JSON dataset at: {dataset_json}")
    df_10k = pd.read_json(dataset_json)
else:
    # Attempt 2: Auto-discover and merge zip bundles
    print("Searching for candidate bundles to auto-unify...")
    zip_9k = find_file("dataset_9k_bundle.zip")
    zip_6k = find_file("dataset_6k_balanced_bundle.zip")
    zip_3k = find_file("candidates_6k_bundle.zip")
    
    parts = []
    cols = [
        'id', 'question_id', 'category', 'question', 'scholar', 'fatwa_id',
        'title', 'question_text', 'answer_text', 'embedding_score',
        'reranker_score', 'rank_in_scholar', 'dataset_split',
        'gpt_verdict', 'gpt_confidence', 'gpt_explanation'
    ]
    
    if zip_9k and zip_9k.exists():
        print(f"  • Reading {zip_9k}")
        with zipfile.ZipFile(zip_9k) as z:
            name = "dataset_9k_full_raw.csv" if "dataset_9k_full_raw.csv" in z.namelist() else "dataset_9k_labeled.csv"
            parts.append(pd.read_csv(z.open(name)))
            
    if zip_6k and zip_6k.exists():
        print(f"  • Reading {zip_6k}")
        with zipfile.ZipFile(zip_6k) as z:
            name = "dataset_6k_balanced.csv" if "dataset_6k_balanced.csv" in z.namelist() else z.namelist()[0]
            parts.append(pd.read_csv(z.open(name)))
            
    if zip_3k and zip_3k.exists():
        print(f"  • Reading {zip_3k}")
        with zipfile.ZipFile(zip_3k) as z:
            if "official_labeled_3k.csv" in z.namelist():
                parts.append(pd.read_csv(z.open("official_labeled_3k.csv")))
                
    if parts:
        for p in parts:
            for c in cols:
                if c not in p.columns:
                    p[c] = None
        df_10k = pd.concat([p[cols] for p in parts], ignore_index=True)
        df_10k = df_10k.drop_duplicates(subset=['question', 'scholar', 'fatwa_id']).reset_index(drop=True)
        df_10k['id'] = range(1, len(df_10k) + 1)
        print(f"✓ Unified {len(df_10k):,} unique candidates across {df_10k['question'].nunique():,} questions.")

assert df_10k is not None and len(df_10k) > 0, "Could not locate 10k dataset or candidate bundles. Please upload dataset_10k_multi_scholar.csv or the dataset bundles."

print(f"\n📊 Dataset Summary:")
print(f"  • Total Candidates: {len(df_10k):,}")
print(f"  • Unique Questions:  {df_10k['question'].nunique():,}")
print(f"  • Scholars:          {df_10k['scholar'].nunique():,}")
print(f"  • Verdict Counts:")
for verdict, cnt in df_10k['gpt_verdict'].value_counts().items():
    print(f"    - {verdict:16s}: {cnt:,} ({cnt/len(df_10k)*100:.1f}%)")


✓ Found unified 10k dataset at: /kaggle/input/datasets/abderrahimzineddine/multi-scholar-10k-dataset/dataset_10k_multi_scholar.csv

📊 Dataset Summary:
  • Total Candidates: 10,044
  • Unique Questions:  942
  • Scholars:          7
  • Verdict Counts:
    - does-not-answer : 7,438 (74.1%)
    - answers-it      : 2,606 (25.9%)


In [4]:
# 4. Load Mizan-Rerank-v3-Turbo Cross-Encoder
from sentence_transformers import CrossEncoder

RERANKER_V3_NAME = "ALJIACHI/Mizan-Rerank-v3-Turbo"
MAX_LENGTH = 2048  # ModernBERT supports up to 8192; 2048 covers virtually all fatwa texts

print(f"Loading {RERANKER_V3_NAME} on [{device}]...")
t0 = time.perf_counter()

reranker_v3 = CrossEncoder(
    RERANKER_V3_NAME,
    max_length=MAX_LENGTH,
    trust_remote_code=True,
    device=device,
)
print(f"✓ Model loaded successfully in {time.perf_counter() - t0:.2f}s!")

# Quick sanity check / warmup
warmup_query = "ما حكم قراءة سورة الكهف يوم الجمعة؟"
warmup_docs = [
    "يستحب قراءة سورة الكهف يوم الجمعة وليلتها لورود الأحاديث في فضل ذلك عن النبي صلى الله عليه وسلم.",
    "صلاة الاستسقاء سنة مؤكدة إذا أجدبت الأرض وانحبس المطر."
]
test_scores = reranker_v3.predict([[warmup_query, doc] for doc in warmup_docs])
print("\nWarmup Test:")
print(f"  • Relevant doc score:   {test_scores[0]:.4f}")
print(f"  • Irrelevant doc score: {test_scores[1]:.4f}")
assert test_scores[0] > test_scores[1], "Sanity check failed: relevant doc must score higher than irrelevant doc."
print("✓ Warmup passed! Model is ready for batch scoring.")


Loading ALJIACHI/Mizan-Rerank-v3-Turbo on [cuda]...


modules.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/225 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer_config.json:   0%|          | 0.00/1.75k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.88M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

✓ Model loaded successfully in 15.57s!

Warmup Test:
  • Relevant doc score:   0.1410
  • Irrelevant doc score: 0.0066
✓ Warmup passed! Model is ready for batch scoring.


In [5]:
# 5. Mass Re-Scoring of the 10,044 Candidates with Mizan-Rerank-v3-Turbo
from tqdm.auto import tqdm

# Construct candidate text: "title — answer_text"
df_10k['embed_text'] = df_10k['title'].fillna('').astype(str) + ' — ' + df_10k['answer_text'].fillna('').astype(str)

pairs = [[str(q), str(t)] for q, t in zip(df_10k['question'], df_10k['embed_text'])]

BATCH_SIZE = 64 if device == "cuda" else 16
print(f"Executing batch reranking on {len(pairs):,} pairs with batch_size={BATCH_SIZE}...")

t0 = time.perf_counter()
scores_v3 = reranker_v3.predict(
    pairs,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
)
total_time = time.perf_counter() - t0
throughput = len(pairs) / total_time

df_10k['reranker_v3_score'] = np.round(scores_v3, 4)

print(f"\n⚡ Reranking Complete!")
print(f"  • Total Pairs Scored: {len(pairs):,}")
print(f"  • Total Time:         {total_time:.2f} seconds ({total_time/60:.2f} min)")
print(f"  • Throughput:         {throughput:.1f} pairs/second")
print(f"  • Avg Latency/Pair:   {total_time/len(pairs)*1000:.2f} ms")

# Score distribution comparison
print("\n📊 Score Distribution by Ground Truth Verdict:")
stats_v2 = df_10k.groupby('gpt_verdict')['reranker_score'].agg(['mean', 'median', 'std'])
stats_v3 = df_10k.groupby('gpt_verdict')['reranker_v3_score'].agg(['mean', 'median', 'std'])
stats_emb = df_10k.groupby('gpt_verdict')['embedding_score'].agg(['mean', 'median', 'std'])

summary_scores = pd.DataFrame({
    'Qwen 0.6B Median': stats_emb['median'],
    'Mizan V2 Median': stats_v2['median'],
    'Mizan V3-Turbo Median': stats_v3['median'],
    'V2 Spread (Pos - Neg)': stats_v2.loc['answers-it', 'median'] - stats_v2.loc['does-not-answer', 'median'],
    'V3-Turbo Spread (Pos - Neg)': stats_v3.loc['answers-it', 'median'] - stats_v3.loc['does-not-answer', 'median']
})
display(summary_scores)


Executing batch reranking on 10,044 pairs with batch_size=64...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]


⚡ Reranking Complete!
  • Total Pairs Scored: 10,044
  • Total Time:         152.81 seconds (2.55 min)
  • Throughput:         65.7 pairs/second
  • Avg Latency/Pair:   15.21 ms

📊 Score Distribution by Ground Truth Verdict:


,Qwen 0.6B Median,Mizan V2 Median,Mizan V3-Turbo Median,V2 Spread (Pos - Neg),V3-Turbo Spread (Pos - Neg)
gpt_verdict,,,,,
answers-it,0.628562,0.912876,0.24370,0.488926,0.17735
does-not-answer,0.497782,0.423950,0.06635,0.488926,0.17735


In [6]:
# 6. Side-by-Side Benchmark: Embedder vs Mizan V2 vs Mizan V3-Turbo
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate_ranking(df, query_col, score_col, target_col, positive_label='answers-it', ks=[1, 3, 5, 10]):
    queries = df[query_col].unique()
    hits = {f'Hit@{k}': [] for k in ks}
    precisions = {f'P@{k}': [] for k in ks}
    recalls = {f'R@{k}': [] for k in ks}
    ndcgs = {f'NDCG@{k}': [] for k in ks}
    mrrs = []
    maps = []
    
    binary_targets = (df[target_col] == positive_label).astype(int)
    auc_roc = roc_auc_score(binary_targets, df[score_col])
    auc_pr = average_precision_score(binary_targets, df[score_col])
    
    valid_q_count = 0
    
    for q in queries:
        group = df[df[query_col] == q]
        labels = (group[target_col] == positive_label).astype(int).values
        n_pos = np.sum(labels)
        if n_pos == 0:
            continue
        valid_q_count += 1
        
        # Sort by score descending (stable sort)
        order = np.argsort(-group[score_col].values, kind='mergesort')
        sorted_labels = labels[order]
        
        # MRR
        pos_ranks = np.where(sorted_labels == 1)[0]
        mrrs.append(1.0 / (pos_ranks[0] + 1) if len(pos_ranks) > 0 else 0.0)
        
        # MAP
        running_pos = np.cumsum(sorted_labels)
        ranks = np.arange(1, len(sorted_labels) + 1)
        precisions_at_i = running_pos / ranks
        ap = np.sum(precisions_at_i * sorted_labels) / n_pos
        maps.append(ap)
        
        # Hit@k, P@k, R@k, NDCG@k
        ideal_sorted = np.sort(labels)[::-1]
        for k in ks:
            top_k_labels = sorted_labels[:k]
            hits[f'Hit@{k}'].append(1.0 if np.sum(top_k_labels) > 0 else 0.0)
            precisions[f'P@{k}'].append(np.mean(top_k_labels) if len(top_k_labels) > 0 else 0.0)
            recalls[f'R@{k}'].append(np.sum(top_k_labels) / n_pos)
            
            discounts = np.log2(np.arange(2, len(top_k_labels) + 2))
            dcg = np.sum(top_k_labels / discounts)
            ideal_top_k = ideal_sorted[:k]
            ideal_discounts = np.log2(np.arange(2, len(ideal_top_k) + 2))
            idcg = np.sum(ideal_top_k / ideal_discounts)
            ndcgs[f'NDCG@{k}'].append((dcg / idcg) if idcg > 0 else 0.0)
            
    res = {
        'Evaluated Queries': valid_q_count,
        'MRR': np.mean(mrrs),
        'MAP': np.mean(maps),
        'ROC-AUC': auc_roc,
        'PR-AUC': auc_pr
    }
    for k in ks:
        res[f'Hit@{k}'] = np.mean(hits[f'Hit@{k}'])
        res[f'P@{k}'] = np.mean(precisions[f'P@{k}'])
        res[f'R@{k}'] = np.mean(recalls[f'R@{k}'])
        res[f'NDCG@{k}'] = np.mean(ndcgs[f'NDCG@{k}'])
        
    return res

print("Computing benchmark metrics across all models...")
metrics_emb = evaluate_ranking(df_10k, 'question', 'embedding_score', 'gpt_verdict')
metrics_v2  = evaluate_ranking(df_10k, 'question', 'reranker_score', 'gpt_verdict')
metrics_v3  = evaluate_ranking(df_10k, 'question', 'reranker_v3_score', 'gpt_verdict')

comparison_df = pd.DataFrame([metrics_emb, metrics_v2, metrics_v3], index=[
    'Qwen 0.6B Embedder',
    'Mizan-Rerank-V2',
    'Mizan-Rerank-v3-Turbo'
]).T

comparison_df['Δ (V3 vs V2)'] = comparison_df['Mizan-Rerank-v3-Turbo'] - comparison_df['Mizan-Rerank-V2']
comparison_df['Δ (V3 vs Embedder)'] = comparison_df['Mizan-Rerank-v3-Turbo'] - comparison_df['Qwen 0.6B Embedder']

print("\n🏆 Side-by-Side Multi-Scholar Retrieval & Reranking Leaderboard:")
display(comparison_df.style.format('{:.4f}'))


Computing benchmark metrics across all models...

🏆 Side-by-Side Multi-Scholar Retrieval & Reranking Leaderboard:


,Qwen 0.6B Embedder,Mizan-Rerank-V2,Mizan-Rerank-v3-Turbo,Δ (V3 vs V2),Δ (V3 vs Embedder)
Evaluated Queries,641.0000,641.0000,641.0000,0.0000,0.0000
MRR,0.8413,0.8431,0.8549,0.0118,0.0136
MAP,0.7862,0.7986,0.8088,0.0103,0.0227
ROC-AUC,0.8277,0.9110,0.8684,-0.0426,0.0408
PR-AUC,0.6104,0.7907,0.7607,-0.0300,0.1503
Hit@1,0.7473,0.7395,0.7832,0.0437,0.0359
P@1,0.7473,0.7395,0.7832,0.0437,0.0359
R@1,0.2465,0.2453,0.2666,0.0213,0.0200
NDCG@1,0.7473,0.7395,0.7832,0.0437,0.0359
Hit@3,0.9267,0.9423,0.9158,-0.0265,-0.0109


In [7]:
# 7. Live Multi-Scholar Corpus Retrieval + V3 Reranking Demo
# Downloads precomputed embeddings and corpus if available, runs full pipeline
from huggingface_hub import hf_hub_download
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

CORPUS_CSV = find_file("multi_scholar_clean.csv")
EMBEDDINGS_PT = find_file("qwen3_06b_multi_scholar_embeddings.pt")

if CORPUS_CSV is None or EMBEDDINGS_PT is None:
    print("Downloading Multi-Scholar Corpus & Precomputed Embeddings from Hugging Face...")
    try:
        CORPUS_CSV = hf_hub_download(
            repo_id="ZenoAbderrahim/binbaz_fatwas_rag",
            filename="multi_scholar_clean.csv",
            repo_type="dataset"
        )
        EMBEDDINGS_PT = hf_hub_download(
            repo_id="ZenoAbderrahim/binbaz_fatwas_rag",
            filename="qwen3_06b_multi_scholar_embeddings.pt",
            repo_type="dataset"
        )
        print("✓ Download completed!")
    except Exception as e:
        print(f"Notice: Could not download full corpus ({e}). Skipping live demo cell.")

if CORPUS_CSV and EMBEDDINGS_PT:
    df_corpus = pd.read_csv(CORPUS_CSV)
    corpus_tensor = torch.load(EMBEDDINGS_PT, map_location="cpu")
    if isinstance(corpus_tensor, dict):
        corpus_tensor = corpus_tensor.get("embeddings", corpus_tensor.get("tensor"))
    if not isinstance(corpus_tensor, torch.Tensor):
        corpus_tensor = torch.tensor(corpus_tensor)
    corpus_tensor = F.normalize(corpus_tensor.float(), p=2, dim=1)
    
    print(f"Corpus loaded: {len(df_corpus):,} fatwas across {df_corpus['source'].nunique()} scholars.")
    
    embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)
    
    def live_search(query: str, top_k_retrieval=20, top_n_rerank=5):
        q_emb = embedder.encode(query, convert_to_tensor=True, device=device).cpu()
        q_emb = F.normalize(q_emb.float().unsqueeze(0), p=2, dim=1)
        sims = (q_emb @ corpus_tensor.T).squeeze(0)
        
        top_indices = torch.topk(sims, top_k_retrieval).indices.numpy()
        candidates = df_corpus.iloc[top_indices].copy().reset_index(drop=True)
        candidates['retrieval_score'] = sims[top_indices].numpy()
        
        # Rerank with V3-Turbo
        cand_texts = [f"{r['title']} — {r['answer']}" for _, r in candidates.iterrows()]
        rerank_scores = reranker_v3.predict([[query, t] for t in cand_texts])
        candidates['v3_score'] = rerank_scores
        
        return candidates.sort_values('v3_score', ascending=False).head(top_n_rerank)
        
    demo_q = "ما كفارة اليمين ومتى يشرع الصيام فيها؟"
    print(f"\n🔍 Demo Query: {demo_q}")
    results = live_search(demo_q, top_k_retrieval=20, top_n_rerank=5)
    
    for idx, (_, row) in enumerate(results.iterrows(), 1):
        print(f"\n[{idx}] {row['source']} | V3 Score: {row['v3_score']:.4f} (Retrieval: {row['retrieval_score']:.4f})")
        print(f"    Title: {row['title']}")
        print(f"    Excerpt: {str(row['answer'])[:180]}...")


multi_scholar_clean.csv: reconstructing file:   0%|          |  0.00B /  131MB            

multi_scholar_clean.csv: downloading bytes:           |  0.00B            

qwen3_06b_multi_scholar_embeddings.pt: reconstructing file:   0%|          |  0.00B /  164MB            

qwen3_06b_multi_scholar_embeddings.pt: downloading bytes:           |  0.00B            

✓ Download completed!
Corpus loaded: 40,062 fatwas across 7 scholars.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

... (9 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]


🔍 Demo Query: ما كفارة اليمين ومتى يشرع الصيام فيها؟

[1] Ibn Baz | V3 Score: 0.5933 (Retrieval: 0.5902)
    Title: ما هي كفارة اليمين؟
    Excerpt: كفارة اليمين أوضحها الله في القرآن في سورة المائدة في قوله سبحانه: لا يُؤَاخِذُكُمُ اللَّهُ بِاللَّغْوِ فِي أَيْمَانِكُمْ وَلَكِنْ يُؤَاخِذُكُمْ بِمَا عَقَّدْتُمُ الأَيْمَانَ فَكَف...

[2] Ibn Baz | V3 Score: 0.2310 (Retrieval: 0.5782)
    Title: توضيح كيفية كفارة اليمين
    Excerpt: كفارة اليمين مثل ما قال الله جل وعلا: فَكَفَّارَتُهُ إِطْعَامُ عَشَرَةِ مَسَاكِينَ مِنْ أَوْسَطِ مَا تُطْعِمُونَ أَهْلِيكُمْ أَوْ كِسْوَتُهُمْ أَوْ تَحْرِيرُ رَقَبَةٍ [المائدة:89] ...

[3] Ibn Baz | V3 Score: 0.1973 (Retrieval: 0.5903)
    Title: صفة كفارة اليمين
    Excerpt: إذا كان عليه كفارة يمين، أو كفارة ظهار، فكفارة اليمين يخير بين ثلاثة: إما إطعام عشرة مساكين، لكل مسكين نصف صاع من قوت البلد: تمرًا، أو أرزًا، أو نحو ذلك، عشرة، كل واحد نصف صاع، كيل...

[4] Saleh Al-Fawzan | V3 Score: 0.1730 (Retrieval: 0.6401)
    Title: كفارة اليمين.
    Excerpt: الجواب

In [8]:
# 8. Export Rescored Dataset & Create Download Archive
OUT_DIR = Path("/kaggle/working/mizan_v3_turbo_outputs") if Path("/kaggle/working").exists() else Path("mizan_v3_turbo_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Export rescored 10k datasets
csv_out = OUT_DIR / "dataset_10k_v3_turbo_rescored.csv"
json_out = OUT_DIR / "dataset_10k_v3_turbo_rescored.json"

export_cols = [
    'id', 'question_id', 'category', 'question', 'scholar', 'fatwa_id',
    'title', 'question_text', 'answer_text',
    'embedding_score', 'reranker_score', 'reranker_v3_score',
    'rank_in_scholar', 'dataset_split', 'gpt_verdict', 'gpt_confidence', 'gpt_explanation'
]
cols_to_use = [c for c in export_cols if c in df_10k.columns]

df_10k[cols_to_use].to_csv(csv_out, index=False)
df_10k[cols_to_use].to_json(json_out, orient="records", force_ascii=False, indent=2)
print(f"✓ Saved rescored dataset:\n  - {csv_out}\n  - {json_out}")

# 2. Export benchmark metrics JSON
metrics_out = OUT_DIR / "v3_turbo_vs_v2_metrics.json"
with open(metrics_out, "w", encoding="utf-8") as f:
    json.dump({
        "model_evaluated": RERANKER_V3_NAME,
        "baseline_reranker": "ALJIACHI/Mizan-Rerank-V2",
        "embedder": "Qwen/Qwen3-Embedding-0.6B",
        "dataset_size": len(df_10k),
        "unique_questions": int(df_10k['question'].nunique()),
        "metrics_summary": comparison_df.to_dict()
    }, f, ensure_ascii=False, indent=2)
print(f"✓ Saved metrics JSON: {metrics_out}")

# 3. Create zip package for easy 1-click download in Kaggle
ZIP_PATH = Path("/kaggle/working/mizan_v3_turbo_evaluation_bundle.zip") if Path("/kaggle/working").exists() else Path("mizan_v3_turbo_evaluation_bundle.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT_DIR.iterdir():
        z.write(f, arcname=f.name)

print(f"\n📦 Download Package Ready:")
print(f"  • Archive: {ZIP_PATH} ({ZIP_PATH.stat().st_size / (1024*1024):.2f} MB)")
print("  • You can download this bundle directly from Kaggle's working directory file browser!")


✓ Saved rescored dataset:
  - /kaggle/working/mizan_v3_turbo_outputs/dataset_10k_v3_turbo_rescored.csv
  - /kaggle/working/mizan_v3_turbo_outputs/dataset_10k_v3_turbo_rescored.json
✓ Saved metrics JSON: /kaggle/working/mizan_v3_turbo_outputs/v3_turbo_vs_v2_metrics.json

📦 Download Package Ready:
  • Archive: /kaggle/working/mizan_v3_turbo_evaluation_bundle.zip (3.90 MB)
  • You can download this bundle directly from Kaggle's working directory file browser!
